In [ ]:
# 必要なものを用意
from transformers import pipeline
distilled_ckpt = "kirapika2/distilbert-base-uncased-distilled-clinc"
pipe = pipeline("text-classification", model=distilled_ckpt)

In [ ]:
# モデルのアテンション重みをプロット
import matplotlib.pyplot as plt

state_dict = pipe.model.state_dict()
weights = state_dict["distilbert.transformer.layer.0.attention.out_lin.weight"]
plt.hist(weights.flatten().cpu().numpy(), bins=250, range=(-0.3, 0.3), edgecolor="C0")
plt.show()

In [ ]:
# スケール係数計算
zero_point = 0
scale = (weights.max() - weights.min()) / (127 - (-128))

In [ ]:
# 量子化テンソルを計算
(weights / scale + zero_point).clamp(-128, 127).round().char()

In [ ]:
# torch.quantize_per_tensor を用いる方法
import torch
from torch import quantize_per_tensor

dtype = torch.qint8
quantized_weights = quantize_per_tensor(weights, scale=scale, zero_point=zero_point, dtype=dtype)
quantized_weights.int_repr()

In [ ]:
## 量子化テンソルを戻してプロット
dequantized_weights = quantized_weights.dequantize()
plt.hist(dequantized_weights.flatten().cpu().numpy(), bins=250, range=(-0.3, 0.3), edgecolor="C0")
plt.show()